# KCV-TTS Kaggle Wheel Builder (Source Build)
Notebook ini fokus untuk build wheel dari source (`causal-conv1d` + `mamba-ssm`) di Kaggle, lalu export wheel agar bisa dipakai ulang tanpa compile.

## 1) Konfigurasi Minimal
Isi repo URL/branch, lalu jalankan cell setup dan build.

In [ ]:
import os
from pathlib import Path

# Repo kamu
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"

# Workspace di Kaggle
WORKDIR = Path('/kaggle/temp')
VENV_DIR = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'

# Output wheel
WHEEL_DIR = Path('/kaggle/working/wheelhouse')
WHEEL_ARCHIVE = Path('/kaggle/working/wheelhouse_mamba_torch260_cu124.tar.gz')

assert REPO_URL != "https://github.com/<username>/<repo>.git", "Isi REPO_URL dulu"

os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WORKDIR'] = str(WORKDIR)
os.environ['VENV_DIR'] = str(VENV_DIR)
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['WHEEL_DIR'] = str(WHEEL_DIR)
os.environ['WHEEL_ARCHIVE'] = str(WHEEL_ARCHIVE)

print('Config OK')
print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('WHEEL_DIR =', WHEEL_DIR)
print('WHEEL_ARCHIVE =', WHEEL_ARCHIVE)

Config OK
SRC_METADATA_CSV = /kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv


## 2) Setup Venv + Clone Repo + Install Base Dependencies

In [3]:
%%bash
apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils > /dev/null
python3.11 --version

Python 3.11.15


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%bash
set -euo pipefail
mkdir -p /kaggle/temp
cd /kaggle/temp

# Isolate from host/site customizations that can break venv bootstrap.
export PYTHONNOUSERSITE=1
unset PYTHONPATH || true
unset PYTHONHOME || true

# Require Python 3.11 to match local ABI/wheels exactly.
if command -v python3.11 >/dev/null 2>&1; then
  PY311=python3.11
else
  echo "python3.11 tidak tersedia di runtime ini. Ganti Kaggle image/runtime yang punya Python 3.11."
  exit 1
fi

# Build venv (fallback to virtualenv if stdlib venv fails).
if ! "$PY311" -m venv .venv; then
  "$PY311" -m pip install --upgrade pip
  "$PY311" -m pip install virtualenv
  "$PY311" -m virtualenv .venv
fi

source .venv/bin/activate
python - <<'PY'
import sys
assert sys.version_info[:2] == (3, 11), f"Butuh Python 3.11, dapat: {sys.version}"
print('python =', sys.version.split()[0])
PY

python -m pip install --upgrade pip setuptools wheel wrapt

# Clone repo
: "${REPO_BRANCH:=main}"
: "${REPO_URL:?REPO_URL is not set. Run Cell 3 (config) first.}"
if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" kcv-tts

cd kcv-tts

# Match stack exactly: torch 2.6/cu124 + project deps
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps

python = 3.11.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 79.0.1
    Uninstalling setuptools-79.0.1:
      Successfully uninstalled setuptools-79.0.1
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing met

Cloning into 'kcv-tts'...


In [ ]:
%%bash
# Build mamba stack from source, export wheels, then install from local prebuilt wheels.
set -euo pipefail

VENV=/kaggle/temp/.venv
WHEEL_DIR=/kaggle/working/wheelhouse

echo "=== Step 1: Uninstall old packages ==="
"$VENV/bin/pip" uninstall -y causal-conv1d mamba-ssm 2>/dev/null || true
"$VENV/bin/pip" cache purge || true

echo "=== Step 2: Check torch ABI ==="
source "$VENV/bin/activate"
python - <<'PY'
import torch
print(f"torch version: {torch.__version__}")
print(f"torch cuda: {torch.version.cuda}")
print(f"torch cxx11abi: {torch._C._GLIBCXX_USE_CXX11_ABI}")
PY

echo "=== Step 3: Deep cleanup .so files ==="
python - <<'PY'
import site, glob, os, shutil
for d in site.getsitepackages():
    for pattern in ["causal_conv1d*", "mamba_ssm*", "causal_conv1d_cuda*", "selective_scan_cuda*"]:
        for p in glob.glob(os.path.join(d, pattern)):
            try:
                if os.path.isdir(p):
                    shutil.rmtree(p, ignore_errors=True)
                else:
                    os.remove(p)
                print(f"removed: {p}")
            except Exception:
                pass
PY

echo "=== Step 4: Install build tools ==="
apt-get update -qq && apt-get install -y gcc g++ build-essential python3.11-dev > /dev/null 2>&1

echo "=== Step 5: Build wheels from source (ABI False) ==="
source "$VENV/bin/activate"
mkdir -p "$WHEEL_DIR"

# Kaggle torch2.6.0+cu124 uses old C++ ABI (cxx11abi=False).
export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export CFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export LDFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"

# Keep CUDA runtime paths visible at build and runtime.
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:/usr/local/cuda-12.4/lib64:${LD_LIBRARY_PATH:-}"

echo "Building wheel causal-conv1d==1.6.1 ..."
pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" causal-conv1d==1.6.1

echo "Building wheel mamba-ssm==2.3.1 ..."
pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" mamba-ssm==2.3.1

echo "=== Step 6: Install from local prebuilt wheels ==="
ls -lh "$WHEEL_DIR"
pip install --no-deps --no-cache-dir --force-reinstall "$WHEEL_DIR"/causal_conv1d-*.whl "$WHEEL_DIR"/mamba_ssm-*.whl

echo "=== Step 7: Verify import ==="
python - <<'PY'
import sys
import torch

try:
    import causal_conv1d
    print("OK causal_conv1d imported")
except Exception as e:
    print(f"FAIL causal_conv1d import: {e}")
    sys.exit(1)

try:
    import mamba_ssm
    print("OK mamba_ssm imported")
except Exception as e:
    print(f"FAIL mamba_ssm import: {e}")
    sys.exit(1)

print(f"OK torch ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")
PY

echo "=== Step 8: Pack wheels for download ==="
ARCHIVE=/kaggle/working/wheelhouse_mamba_torch260_cu124.tar.gz
tar -czf "$ARCHIVE" -C /kaggle/working wheelhouse
ls -lh "$ARCHIVE"

echo "Done. Download wheelhouse from /kaggle/working and upload as Kaggle Dataset for future reuse."

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.3/287.3 MB 27.4 MB/s  0:00:10


In [ ]:
%%bash
set -euo pipefail

# Optional reuse cell for next runs:
# 1) Upload /kaggle/working/wheelhouse as a Kaggle Dataset
# 2) Attach that dataset to a new notebook run
# 3) Set WHEEL_SRC_DIR to the mounted dataset path, then run this cell

VENV=/kaggle/temp/.venv
WHEEL_SRC_DIR="${WHEEL_SRC_DIR:-/kaggle/input/your-wheelhouse-dataset/wheelhouse}"

if [ ! -d "$WHEEL_SRC_DIR" ]; then
  echo "Wheel source directory not found: $WHEEL_SRC_DIR"
  echo "Set WHEEL_SRC_DIR to your mounted Kaggle dataset wheelhouse path."
  exit 0
fi

source "$VENV/bin/activate"
ls -lh "$WHEEL_SRC_DIR"

pip install --no-cache-dir --force-reinstall \
  "$WHEEL_SRC_DIR"/causal_conv1d-*.whl \
  "$WHEEL_SRC_DIR"/mamba_ssm-*.whl

python - <<'PY'
import torch, causal_conv1d, mamba_ssm
print('OK torch', torch.__version__, 'abi', torch._C._GLIBCXX_USE_CXX11_ABI)
print('OK causal_conv1d', getattr(causal_conv1d, '__version__', 'n/a'))
print('OK mamba_ssm', getattr(mamba_ssm, '__version__', 'n/a'))
PY